# Stretch - Deploy as an Agent on AgentCore Runtime

In this notebook you build the travel assistant as a Strands agent that selects its own tools, run it locally, and deploy it to Amazon Bedrock AgentCore Runtime.

## Contents

1. [Introduction](#intro)
2. [Setup](#setup)
3. [Pipeline versus agent](#s1)
4. [Define the tools](#s2)
5. [Build the Strands agent](#s3)
6. [Deploy with the AgentCore CLI](#s4)
7. [Invoke the agent](#s5)
8. [Try it](#try-it)
9. [Clean up](#clean-up)
10. [Conclusion and key takeaways](#conclusion)
11. [Next steps](#next-steps)

## 1. Introduction <a id="intro"></a>

In Module 6 you deployed the travel assistant as one Lambda function with two request types, `ask` and `check_expense`. In that design, your code defines the flow: every `ask` request retrieves policy passages and then calls the chat model, in the same order.

Some employee messages combine more than one task. Consider:

> "Was my $340 Chicago hotel within policy, and can I expense the minibar?"

This message contains an expense check (the hotel) and a policy question (the minibar). The Lambda function accepts one request type per call, so the calling application would have to separate the two parts. An **agent** handles this differently: the model receives a set of **tools** and decides which tools to call, in what order, and when it has enough information to answer.

| | The Lambda function | An agent |
|:--|:--|:--|
| Who decides the steps | Your code | The model |
| Suited to | Well-defined, predictable processes | Varied requests that combine several tasks |

AWS Prescriptive Guidance recommends [deterministic execution logic unless AI is needed](https://docs.aws.amazon.com/prescriptive-guidance/latest/agentic-ai-security/best-practices-system-design.html), so use an agent where the requests need it.

In this notebook you build the agent with [Strands Agents](https://strandsagents.com/), an open source SDK for building agents, and deploy it to [Amazon Bedrock AgentCore Runtime](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/agents-tools-runtime.html), a secure, serverless hosting environment for AI agents and tools.

## 2. Setup <a id="setup"></a>

The agent's code is in [`app/agent`](../app/agent): `travel_agent.py` defines the tools and the agent, and `main.py` is the entrypoint that AgentCore Runtime calls. Run the cell below to import it, create the clients, and check that the AgentCore command line interface (CLI) is installed.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
import tempfile
import time
import uuid

import boto3

sys.path.insert(0, "../app/agent")
import travel_agent

# The AWS Region from your AWS configuration (or the AWS_REGION environment variable).
REGION = boto3.session.Session().region_name
ACCOUNT_ID = boto3.client("sts").get_caller_identity()["Account"]

bedrock_runtime = boto3.client("bedrock-runtime", region_name=REGION)
cloudformation = boto3.client("cloudformation", region_name=REGION)
agentcore_runtime = boto3.client("bedrock-agentcore", region_name=REGION)  # Invoke deployed agents

# Set in travel_agent.py. The agent model is Claude Sonnet 5 through a global cross-Region
# inference profile ("global." prefix); expense checks use gpt-oss-120b in this Region (no prefix).
print(f"Region:          {REGION}")
print(f"Agent model:     {travel_agent.AGENT_MODEL_ID}")
print(f"Expense model:   {travel_agent.EXPENSE_MODEL_ID}")
print(f"Knowledge base:  {travel_agent.KNOWLEDGE_BASE_ID}")

if shutil.which("agentcore"):
    print("AgentCore CLI:  ", subprocess.run(["agentcore", "--version"], capture_output=True, text=True).stdout.strip())
else:
    print("The AgentCore CLI is not installed. Install it with: npm install -g @aws/agentcore")

## 3. Pipeline versus agent <a id="s1"></a>

Start with the pipeline approach from Modules 3 and 6: retrieve policy passages for the question, then make one model call. Run the cell below to send it the mixed request.

In [ ]:
PIPELINE_PROMPT = (
    "You are the travel assistant for AnyCompany. Answer using only the AnyCompany policy passages "
    "inside the <policy> tags, and name the policy documents you used. If the passages do not "
    "answer the question, say so. Write plain text, concisely."
)


def pipeline_answer(question):
    """The fixed pipeline: one retrieval, then one model call."""
    passages = travel_agent.retrieve(question)
    response = bedrock_runtime.converse(
        modelId=travel_agent.AGENT_MODEL_ID,
        system=[{"text": PIPELINE_PROMPT}],
        messages=[{
            "role": "user",
            "content": [{"text": f"<policy>\n{travel_agent.format_passages(passages)}\n</policy>\n\nQuestion: {question}"}],
        }],
        inferenceConfig={"maxTokens": 2000},
    )
    return "".join(b["text"] for b in response["output"]["message"]["content"] if "text" in b)


MIXED_REQUEST = "Was my $340 Chicago hotel within policy, and can I expense the minibar?"

started = time.time()
print(pipeline_answer(MIXED_REQUEST))
print(f"\n[pipeline: 1 retrieval, 1 model call, {time.time() - started:.1f} s]")

The pipeline answers from the retrieved policy passages, but it does not run the expense check you built in Modules 2 to 5, because the flow defined in code has no step for it. In the next sections, the model decides which steps to run.

## 4. Define the tools <a id="s2"></a>

A **tool** is a function that the model can ask to call. In Strands, a tool is a Python function with the [`@tool` decorator](https://strandsagents.com/docs/user-guide/sdk/tools/custom-tools/); its docstring and type hints become the tool description that the model reads, so write them with the same care as a prompt.

The travel assistant's two capabilities become two tools in [`travel_agent.py`](../app/agent/travel_agent.py):

| Tool | Does | Built in |
|:--|:--|:--|
| `search_policy(query)` | Retrieves policy passages from the knowledge base | Module 3 |
| `check_expense(trip, items)` | Retrieves the rules for each expense type and asks gpt-oss-120b for APPROVED or FLAGGED | Modules 2 to 5 |

Run the cell below to print each tool's description, then call each tool directly, the way the agent will.

In [ ]:
for agent_tool in [travel_agent.search_policy, travel_agent.check_expense]:
    spec = agent_tool.tool_spec
    print(f"Tool: {spec['name']}")
    print(f"  Description: {spec['description'].splitlines()[0]}")
    print(f"  Inputs:      {list(spec['inputSchema']['json']['properties'])}\n")

print("search_policy('minibar'):")
print(travel_agent.search_policy(query="minibar")[:300], "...\n")

print("check_expense(...):")
print(travel_agent.check_expense(
    trip="Chicago, 1 night",
    items=[{"type": "hotel", "description": "1 night, downtown hotel", "amount": 340}],
))

## 5. Build the Strands agent <a id="s3"></a>

A Strands agent combines a model, a system prompt and a list of tools. When you send it a message, Strands runs the [agent loop](https://strandsagents.com/docs/user-guide/sdk/agents/agent-loop/): the model selects tools, Strands runs them and returns the results, until the model can answer. `create_agent` in `travel_agent.py` builds the agent:

```python
Agent(
    model=BedrockModel(model_id=AGENT_MODEL_ID, region_name=REGION),
    system_prompt=SYSTEM_PROMPT,
    tools=[search_policy, check_expense],
)
```

Run the cell below to create the agent locally, in this notebook, and send it the mixed request.

In [ ]:
def run_agent(agent, prompt):
    """Send one message to a local agent and print the answer and the tools it called."""
    calls_before = travel_agent.tool_call_counts(agent)
    cycles_before = agent.event_loop_metrics.cycle_count  # Counts are totals for the agent's lifetime
    started = time.time()
    result = agent(prompt)
    calls_after = travel_agent.tool_call_counts(agent)
    tools_used = {name: count - calls_before.get(name, 0) for name, count in calls_after.items()
                  if count > calls_before.get(name, 0)}
    print(f"Q: {prompt}\n\n{str(result).strip()}\n")
    print(f"[agent: tools used {tools_used}, {agent.event_loop_metrics.cycle_count - cycles_before} model calls, "
          f"{time.time() - started:.1f} s]")
    return result


local_agent = travel_agent.create_agent()
result = run_agent(local_agent, MIXED_REQUEST)

This time the model selected both tools: `check_expense` for the hotel and `search_policy` for the minibar, then combined the results into one answer.

The agent keeps the conversation history between calls. Run the cell below to send a follow-up question.

In [ ]:
result = run_agent(local_agent, "What if the same stay had been in New York?")

## 6. Deploy with the AgentCore CLI <a id="s4"></a>

To make the agent available to applications, you deploy it to AgentCore Runtime with the [AgentCore CLI](https://github.com/aws/agentcore-cli) (`agentcore`). The CLI manages a **project**, the configuration files that describe your agent, and deploys it with the AWS Cloud Development Kit (AWS CDK).

> **Note:** At AWS events, the CLI, Node.js and the CDK bootstrap are already set up in your account. In your own account, install the CLI with `npm install -g @aws/agentcore`, and [bootstrap](https://docs.aws.amazon.com/cdk/v2/guide/bootstrapping.html) each account and Region you deploy to.

Run the cell below to create the project in a temporary folder.

In [ ]:
PROJECT_NAME = "aim317travel"
RUNTIME_NAME = "travel_assistant"
BUILD_DIR = os.path.join(tempfile.gettempdir(), "aim317-agentcore")
PROJECT_DIR = os.path.join(BUILD_DIR, PROJECT_NAME)

if os.path.exists(os.path.join(PROJECT_DIR, "agentcore", "agentcore.json")):
    print(f"Project already exists in {PROJECT_DIR}")
else:
    os.makedirs(BUILD_DIR, exist_ok=True)
    !agentcore create --project-name {PROJECT_NAME} --name {RUNTIME_NAME} --framework Strands --model-provider Bedrock --memory none --skip-git --skip-python-setup --output-dir {BUILD_DIR} --json

print("\nProject configuration:", os.listdir(os.path.join(PROJECT_DIR, "agentcore")))

The CLI created an example agent. Run the cell below to replace it with the travel assistant's code, set the knowledge base ID as an environment variable, and add a policy that allows the agent to query your knowledge base (`bedrock:Retrieve`).

In [ ]:
AGENT_CODE_DIR = os.path.join(PROJECT_DIR, "agent")
os.makedirs(AGENT_CODE_DIR, exist_ok=True)
for file_name in ["main.py", "travel_agent.py", "pyproject.toml"]:
    shutil.copy(os.path.join("../app/agent", file_name), AGENT_CODE_DIR)

extra_permissions = {
    "Version": "2012-10-17",
    "Statement": [
        {"Effect": "Allow", "Action": "bedrock:Retrieve",
         "Resource": f"arn:aws:bedrock:{REGION}:{ACCOUNT_ID}:knowledge-base/{travel_agent.KNOWLEDGE_BASE_ID}"},
    ],
}
with open(os.path.join(PROJECT_DIR, "agentcore", "travel-assistant-policy.json"), "w") as f:
    json.dump(extra_permissions, f, indent=2)

config_path = os.path.join(PROJECT_DIR, "agentcore", "agentcore.json")
with open(config_path) as f:
    config = json.load(f)
config["runtimes"] = [{
    "name": RUNTIME_NAME,
    "build": "CodeZip",          # AgentCore packages the code and its Python packages for you
    "entrypoint": "main.py",
    "codeLocation": "agent/",
    "runtimeVersion": "PYTHON_3_14",
    "networkMode": "PUBLIC",
    "protocol": "HTTP",
    "envVars": [{"name": "KNOWLEDGE_BASE_ID", "value": travel_agent.KNOWLEDGE_BASE_ID}],
    "additionalPolicies": ["../agentcore/travel-assistant-policy.json"],  # Relative to codeLocation
}]
with open(config_path, "w") as f:
    json.dump(config, f, indent=2)

# Where to deploy: this account and Region.
with open(os.path.join(PROJECT_DIR, "agentcore", "aws-targets.json"), "w") as f:
    json.dump([{"name": "default", "account": ACCOUNT_ID, "region": REGION}], f, indent=2)

print(json.dumps(config["runtimes"][0], indent=2))

Next, deploy the project. `agentcore deploy` packages the code and deploys an AWS CloudFormation stack with the agent's IAM role and its AgentCore Runtime. Run the cell below. Deployment takes one to two minutes.

In [ ]:
started = time.time()
deploy = subprocess.run(["agentcore", "deploy", "--yes", "--json"], cwd=PROJECT_DIR, capture_output=True, text=True)
print(f"Deploy finished in {time.time() - started:.0f} seconds")

if deploy.returncode != 0:
    print("The deployment failed. The last lines of the output:\n")
    print((deploy.stdout + deploy.stderr)[-2000:])
else:
    with open(os.path.join(PROJECT_DIR, "agentcore", ".cli", "deployed-state.json")) as f:
        deployed = json.load(f)["targets"]["default"]["resources"]
    AGENT_RUNTIME_ARN = deployed["runtimes"][RUNTIME_NAME]["runtimeArn"]
    print("Stack:        ", deployed["stackName"])
    print("Runtime ARN:  ", AGENT_RUNTIME_ARN)

## 7. Invoke the agent <a id="s5"></a>

Applications call a deployed agent with the [`InvokeAgentRuntime`](https://docs.aws.amazon.com/bedrock-agentcore/latest/APIReference/API_InvokeAgentRuntime.html) API. `main.py` expects a payload of `{"prompt": "..."}` and returns the answer and the tools used.

`runtimeSessionId` identifies the [session](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/runtime-sessions.html). AgentCore Runtime preserves context between invocations with the same session ID, so use a different ID for each conversation.

Run the cell below to send the mixed request, then the follow-up, in one session. The first request takes longer, because AgentCore Runtime starts the session.

In [ ]:
def ask_deployed_agent(prompt, session_id):
    """Send one message to the deployed agent, in the given session."""
    started = time.time()
    response = agentcore_runtime.invoke_agent_runtime(
        agentRuntimeArn=AGENT_RUNTIME_ARN,
        runtimeSessionId=session_id,
        payload=json.dumps({"prompt": prompt}),
    )
    body = json.loads(response["response"].read())
    print(f"Q: {prompt}\n\n{body['answer']}\n")
    print(f"[deployed agent: tools used {body['tools_used']}, {time.time() - started:.1f} s]\n")


session_id = str(uuid.uuid4())
print(f"Session: {session_id}\n")
ask_deployed_agent(MIXED_REQUEST, session_id)
ask_deployed_agent("What if the same stay had been in New York?", session_id)

The follow-up succeeds because both requests used the same session.

## 8. Try it <a id="try-it"></a>

**Build it yourself: a structured answer.** The agent answers in prose, but an expense system needs fields it can process. Strands supports [structured output](https://strandsagents.com/docs/user-guide/sdk/agents/structured-output/): you define the fields with a Pydantic model, pass the model as `structured_output_model`, and read the validated object from `result.structured_output`.

Complete `ExpenseAnswer` in the cell below with three fields, then run it:

- `within_policy` (`bool`): true only if every expense asked about is reimbursable and within policy
- `reasons` (`list[str]`): one short reason per expense
- `policy_documents` (`list[str]`): the policy documents the agent relied on

Give each field a `Field(description=...)`; the descriptions tell the model what each field must contain.

In [ ]:
from pydantic import BaseModel, Field


class ExpenseAnswer(BaseModel):
    """The agent's answer about one or more expenses."""
    within_policy: bool = Field(description="True only if every expense asked about is within policy")
    # TODO: add the reasons and policy_documents fields.


result = travel_agent.create_agent()(MIXED_REQUEST, structured_output_model=ExpenseAnswer)
print(repr(result.structured_output))

**Solution.** Run the cell below to see one way to complete it.

In [ ]:
class ExpenseAnswer(BaseModel):
    """The agent's answer about one or more expenses."""
    within_policy: bool = Field(description="True only if every expense asked about is reimbursable and within policy")
    reasons: list[str] = Field(description="One short reason per expense")
    policy_documents: list[str] = Field(description="The policy documents relied on")


result = travel_agent.create_agent()(MIXED_REQUEST, structured_output_model=ExpenseAnswer)
answer = result.structured_output
print("Within policy:", answer.within_policy)
for reason in answer.reasons:
    print(" -", reason)
print("Policy documents:", answer.policy_documents)

**More to try:** send the follow-up question to the deployed agent in a **new** session (call `ask_deployed_agent` with a new `str(uuid.uuid4())`). What happens, and why?

## 9. Clean up <a id="clean-up"></a>

AgentCore Runtime is [billed for the resources that agent sessions consume](https://aws.amazon.com/bedrock/agentcore/pricing/), so delete the agent when you no longer need it. `agentcore remove all` resets the project configuration, and `agentcore deploy` then deletes the stack. The stack does not delete the agent's log group, so the cell deletes it too.

Run the cell below to delete the deployed agent and its logs.

In [ ]:
!cd {PROJECT_DIR} && agentcore remove all --yes --json
teardown = subprocess.run(["agentcore", "deploy", "--yes", "--json"], cwd=PROJECT_DIR, capture_output=True, text=True)
print("Teardown", "finished" if teardown.returncode == 0 else "failed:\n" + (teardown.stdout + teardown.stderr)[-2000:])

try:
    status = cloudformation.describe_stacks(StackName=f"AgentCore-{PROJECT_NAME}-default")["Stacks"][0]["StackStatus"]
    print("Stack status:", status)
except cloudformation.exceptions.ClientError:
    print("The agent's stack is deleted.")

# The runtime's log group outlives the stack. Delete it too.
logs = boto3.client("logs", region_name=REGION)
for page in logs.get_paginator("describe_log_groups").paginate(
    logGroupNamePrefix=f"/aws/bedrock-agentcore/runtimes/{PROJECT_NAME}_"
):
    for group in page["logGroups"]:
        logs.delete_log_group(logGroupName=group["logGroupName"])
        print("Deleted log group", group["logGroupName"])

## 10. Conclusion and key takeaways <a id="conclusion"></a>

- An **agent** gives a model tools and lets the model decide which to call and in what order. The capabilities you built for the pipeline became tools without changes to their logic.
- In **Strands Agents**, a tool is a Python function with the `@tool` decorator; its docstring and type hints tell the model how to use it.
- **AgentCore Runtime** hosts the agent and keeps context within a session. The **AgentCore CLI** creates, deploys and removes it.
- Use an agent where requests need the model to decide the steps, and a fixed pipeline where the process is well defined.

## 11. Next steps <a id="next-steps"></a>

To extend the agent, you can connect it to existing APIs and AWS services as tools through [AgentCore Gateway](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/gateway.html), retain conversations beyond a session with [AgentCore Memory](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/memory.html), and assess the quality of its answers with [AgentCore Evaluations](https://docs.aws.amazon.com/bedrock-agentcore/latest/devguide/evaluations.html). When you finish the workshop, run the [Cleanup](99_cleanup.ipynb) notebook to delete the other resources you created.